# 🎙 내 목소리로 23개 언어 말하기 · 0원

**Chatterbox Multilingual** (오픈소스 · MIT)로 내 목소리 그대로 영어·스페인어·일본어… 음성을 만들어요.
실험에서 13개 언어 모두 "내 목소리와 닮음" 1등이었던 모델이에요.

**준비물:** 내 목소리 10~30초 (조용한 곳에서, 말소리만)

**런타임 → 런타임 유형 변경 → T4 GPU**를 켜고, 위에서부터 차례로 실행하세요.
설명 페이지: https://www.aicitybuilders.com/dubbing

⚠️ **본인 목소리이거나, 허락받은 목소리만** 쓰세요.

In [ ]:
# ① 설치 (2~4분)
#
# 빨간 글씨(ERROR: pip's dependency resolver ...)가 떠도 정상입니다.
# 설치가 끝나면 세션이 자동으로 한 번 다시 시작됩니다. 그다음 ② 번 셀부터 실행하세요.

# torchvision 은 torch 2.6 과 짝을 맞춰 설치해요 (안 맞으면 ② 번에서 LlamaModel 에러가 나요)
!pip install -q chatterbox-tts "torchvision==0.21.0" "setuptools<81"

import os
FLAG = '/content/.voice_setup_done'
if not os.path.exists(FLAG):
    open(FLAG, 'w').close()
    print('설치 완료. 세션을 다시 시작합니다 — 잠시 뒤 ② 번 셀부터 실행하세요.')
    os.kill(os.getpid(), 9)
else:
    print('준비 완료 — ② 번 셀로 넘어가세요.')

In [ ]:
# ② 목소리 엔진 불러오기 (1~2분, 처음 한 번만 모델을 내려받아요)
import os, subprocess, torch, torchaudio
from IPython.display import Audio, display
from chatterbox.mtl_tts import ChatterboxMultilingualTTS, SUPPORTED_LANGUAGES

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
tts = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE)
OUT = os.environ.get('DUB_OUT', '/content/voice')
os.makedirs(OUT, exist_ok=True)
print(f'준비 완료 · {DEVICE} · 언어 {len(SUPPORTED_LANGUAGES)}개:', ', '.join(sorted(SUPPORTED_LANGUAGES)))

In [ ]:
# ③ 내 목소리 올리기 — 10~30초, 말소리만 깨끗하게
#
# 파일 선택 창이 뜨면 mp3 / wav / m4a / mp4 아무거나 올리세요.
if os.environ.get('DUB_LOCAL') == '1':
    src = os.environ['DUB_VOICE']
else:
    from google.colab import files
    src = next(iter(files.upload()))

VOICE = os.path.join(os.path.dirname(OUT), 'my_voice.wav')      # 결과 폴더와 따로 보관
subprocess.run(['ffmpeg', '-y', '-v', 'error', '-i', src, '-ac', '1', '-ar', '24000', '-t', '30', VOICE], check=True)   # 모노 · 24kHz · 앞 30초
display(Audio(VOICE))

In [ ]:
# ④ 문장 하나 만들어 보기 — 글자와 언어만 바꾸면 돼요
TEXT = "Hi, I'm Jay from Connect AI Lab. This is my own voice, now speaking your language."
LANG = 'en'                 # en 영어 · es 스페인어 · fr 프랑스어 · hi 힌디어 … ②번에 나온 목록

wav = tts.generate(TEXT, language_id=LANG, audio_prompt_path=VOICE,
                   exaggeration=0.5,     # 감정 세기 0.25~1.0 (클수록 생생하게)
                   cfg_weight=0.5)       # 0.3 쯤으로 낮추면 말이 차분하고 느려져요
torchaudio.save(f'{OUT}/{LANG}.wav', wav.cpu(), tts.sr)
display(Audio(f'{OUT}/{LANG}.wav'))

In [ ]:
# ⑤ 여러 언어로 한 번에
SAY = {
    'en': "Hi, I'm Jay from Connect AI Lab. This is my own voice, now speaking your language.",
    'es': 'Hola, soy Jay de Connect AI Lab. Esta es mi propia voz, ahora hablando tu idioma.',
    'fr': "Bonjour, je suis Jay de Connect AI Lab. C'est ma propre voix, qui parle maintenant votre langue.",
    'hi': 'नमस्ते, मैं Connect AI Lab से जय हूँ। यह मेरी अपनी आवाज़ है, जो अब आपकी भाषा में बोल रही है।',
    'zh': '大家好，我是 Connect AI Lab 的 Jay。这是我自己的声音，现在正在说你的语言。',
}
for lang, text in SAY.items():
    wav = tts.generate(text, language_id=lang, audio_prompt_path=VOICE)
    torchaudio.save(f'{OUT}/{lang}.wav', wav.cpu(), tts.sr)
    print(lang, text); display(Audio(f'{OUT}/{lang}.wav'))

In [ ]:
# ⑥ 내려받기 — 만든 음성을 zip 하나로
import shutil
shutil.make_archive(f'{OUT}_all', 'zip', OUT)
if os.environ.get('DUB_LOCAL') != '1':
    from google.colab import files
    files.download(f'{OUT}_all.zip')
print('완료 ✅', sorted(f for f in os.listdir(OUT) if f.endswith('.wav')))

### 팁
- 같은 문장도 실행할 때마다 조금씩 달라요. 마음에 안 들면 그 셀만 다시 실행하세요.
- **일본어**는 한자를 잘 못 읽어요. 히라가나·가타카나로 바꿔 넣으면 훨씬 정확해져요.
- **인도네시아어·베트남어·태국어**는 이 모델이 지원하지 않아요.
- 영상에 쓰면 설명란에 "AI 음성(내 목소리)"이라고 적어 주세요.